# Bronze — trust prices from Yahoo

`landing.trust_prices_yf_raw` → `bronze.trust_prices_yf`. Every column cast to STRING.

This is the table where the cast does real work: Landing holds yfinance's own types
(timestamps, doubles, longs) and Bronze flattens them to text like everything else.

Column names keep Yahoo's TitleCase. Renaming is a transformation and belongs in Silver;
Bronze changes exactly one thing, the type.

One consequence for Silver: `CAST(Date AS STRING)` on a TIMESTAMP yields
`2026-09-01 00:00:00`, not `2026-09-01`. Silver parses it back and derives the month key.

Expected: **35,121 rows across 100 symbols**, matching Landing exactly.

In [0]:
CATALOG = "`index-vs-trust-pipeline`"
SOURCE = f"{CATALOG}.landing.trust_prices_yf_raw"
TARGET = f"{CATALOG}.bronze.trust_prices_yf"

In [0]:
src_columns = spark.table(SOURCE).columns

# Cast whatever arrived. Yahoo's column set moves between instruments -- Capital_Gains
# comes back for the index and not for the trusts -- so a hardcoded list would drop it.
cast_list = ",\n  ".join(f"CAST(`{c}` AS STRING) AS `{c}`" for c in src_columns)
sql = f"CREATE OR REPLACE TABLE {TARGET} AS\nSELECT\n  {cast_list}\nFROM {SOURCE}"

print(f"{len(src_columns)} columns found: {src_columns}\n")
print(sql)

spark.sql(sql)
print(f"\nwrote {TARGET}")

## Verification

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.landing.trust_prices_yf_raw) AS landing_rows,
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf)      AS bronze_rows,
  (SELECT COUNT(DISTINCT symbol) FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf) AS symbols,
  (SELECT MIN(`Date`) FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf)   AS first_bar;

Expect **35,121 / 35,121 / 100 symbols**, earliest bar **1967-12-01 00:00:00**.

The first bar now sorts as text rather than as a date, which is the cast doing its job.

In [0]:
%sql
-- The contract. Any non-STRING column here is a bug.
SELECT COUNT(*)                                              AS columns_total,
       SUM(CASE WHEN data_type <> 'STRING' THEN 1 ELSE 0 END) AS not_string
FROM `index-vs-trust-pipeline`.information_schema.columns
WHERE table_schema = 'bronze' AND table_name = 'trust_prices_yf';

Expect **11 columns, 0 not_string** — no `Capital_Gains` on the trust side.